In [1]:
# Import Libraries
import numpy as np
import pandas as pd

# Dataset
from sklearn.datasets import load_breast_cancer

# Importing Libraries and Evaluation Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# The 5 Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier



# 1. Loading and Preprocessing

In [2]:
# Load and Explore the Dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns= data.feature_names)
df["target"] = data.target

print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum().sum())
print("\nClass distribution:\n", df['target'].value_counts())
print("\nFeature statistics:\n", df.describe())

Shape: (569, 31)

Missing values:
 0

Class distribution:
 target
1    357
0    212
Name: count, dtype: int64

Feature statistics:
        mean radius  mean texture  mean perimeter    mean area  \
count   569.000000    569.000000      569.000000   569.000000   
mean     14.127292     19.289649       91.969033   654.889104   
std       3.524049      4.301036       24.298981   351.914129   
min       6.981000      9.710000       43.790000   143.500000   
25%      11.700000     16.170000       75.170000   420.300000   
50%      13.370000     18.840000       86.240000   551.100000   
75%      15.780000     21.800000      104.100000   782.700000   
max      28.110000     39.280000      188.500000  2501.000000   

       mean smoothness  mean compactness  mean concavity  mean concave points  \
count       569.000000        569.000000      569.000000           569.000000   
mean          0.096360          0.104341        0.088799             0.048919   
std           0.014064          0.05281

In [3]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


* The breast cancer dataset contains 569 samples and 31 columns (30 features + 1 target), with no missing values, making it ready for modeling without imputation.
* The class distribution shows a mild imbalance (357 benign, 212 malignant), which can be addressed using stratified splitting to preserve class proportions in both train and test sets.
* Feature statistics revealed extreme scale differences across variables — for example, mean area ranges up to 2501 while mean smoothness stays below 0.17 — confirming that StandardScaler was necessary to normalize all features to a common scale before applying distance-sensitive algorithms like SVM and k-NN.

In [4]:
# Seperate features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42, stratify= y)

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Traing set size:", X_train_scaled.shape)
print("Test set size:", X_test_scaled.shape)

Traing set size: (455, 30)
Test set size: (114, 30)


# 2. Classification Algorithm Implementation

In [5]:
models = {
    "Logistic Regression":    LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":          DecisionTreeClassifier(random_state=42),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM":                    SVC(kernel="rbf", random_state=42),
    "K-Nearest Neighbors":    KNeighborsClassifier(n_neighbors=5)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred,target_names= data.target_names))


Model: Logistic Regression
Accuracy: 0.9825
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Model: Decision Tree
Accuracy: 0.9123
              precision    recall  f1-score   support

   malignant       0.85      0.93      0.89        42
      benign       0.96      0.90      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.92      0.91       114
weighted avg       0.92      0.91      0.91       114


Model: RandomForestClassifier
Accuracy: 0.9561
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       

**1. Logistic Regression**
* Logistic Regression models the probability of a sample belonging to a class using the sigmoid function and finds a linear decision boundary between classes.
* It is suitable for this dataset because the features, after StandardScaling, are well-separated and the relationship between features and the target is approximately linear.

**2. Decision Tree Classifier**
* A Decision Tree recursively splits data based on feature thresholds, selecting splits that maximize information gain or minimize Gini impurity, forming a tree of if-else decisions.
* It is suitable here as it can capture non-linear relationships between tumour characteristics without requiring feature scaling.

**3. Random Forest Classifier**
* Random Forest builds multiple decision trees on random subsets of data and features, combining their predictions by majority vote to reduce overfitting.
* It is suitable for this dataset due to its ability to handle 30 high-dimensional features while maintaining strong generalisation.

**4. Support Vector Machine(SVM)**
* SVM finds the optimal hyperplane that maximally separates two classes in high-dimensional feature space, using an RBF kernel to handle non-linear boundaries.
* It is highly suitable for this dataset as the features after scaling produce well-separated classes in high-dimensional space.

**5. K-Nearest Neighbors (k-NN)**
* k-NN classifies a sample based on the majority class among its k nearest neighbours in feature space, making no assumptions about the underlying data distribution.
* It is suitable here as the dataset is small enough for efficient distance computation and features are scaled, which is essential for distance-based methods.


# 3. Model Comparison

In [6]:
results_df = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
results_df = results_df.sort_values("Accuracy", ascending= False).reset_index(drop= True)

print("Model Performance Comparison:")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
worst = results_df.iloc[-1]

print(f"\nBest model: {best["Model"]} with accuracy {best["Accuracy"]:.4f}")
print(f"Worst model: {worst["Model"]} with accuracy {worst["Accuracy"]:.4f}")

Model Performance Comparison:
                 Model  Accuracy
   Logistic Regression  0.982456
                   SVM  0.982456
RandomForestClassifier  0.956140
   K-Nearest Neighbors  0.956140
         Decision Tree  0.912281

Best model: Logistic Regression with accuracy 0.9825
Worst model: Decision Tree with accuracy 0.9123


* Among the five classification algorithms evaluated on the breast cancer dataset, Logistic Regression and SVM performed equally best with an accuracy of 98.25%, followed by Random Forest and K-Nearest Neighbors both at 95.61%, and Decision Tree as the worst performer at 91.23%.
* Logistic Regression and SVM excel on this dataset because both rely on finding a clean decision boundary in feature space, which is achievable here due to the well-scaled, high-dimensional features after StandardScaling.
* Decision Tree performed the worst because without pruning it overfits to training data, failing to generalise to unseen samples — reflected in its lowest malignant precision of 85% among all five models.
* Overall, Logistic Regression is the preferred model for this task — it matches SVM in accuracy while being simpler and more interpretable, which is a critical requirement in clinical decision-making contexts like tumour diagnosis.